In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [3]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [ ]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    return df

Arreglar ejercicios de Ponencia

In [5]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [6]:
df = df.apply(fix_ponencia, axis = 1)

In [7]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,enunciados_ejercicios/Mates II/matrices_determ...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [11]:
print(df[df.ejercicio.isna()].info_ejercicio)

2556    MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...
Name: info_ejercicio, dtype: str


In [12]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [14]:
df.sample(3)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
1726,Mates II,Mates II,2000,Reserva 3,Ejercicio 4. Opción A,Geometría,MATEMÁTICAS II. 2000. RESERVA 3. EJERCICIO 4. ...,Halla las coordenadas del punto simétrico del ...,enunciados_ejercicios/Mates II/geometría/Mate...,enunciados_ejercicios/Mates II/geometría/png/...,NaN
641,Mates CCSS,Mates CCSS,2004,Reserva 4,Ejercicio 1 Opción B,Sistemas de Ecuaciones Lineales,SOCIALES II. 2004 RESERVA 4. EJERCICIO 1 OPCIÓN B,x yz  2\n\nSea el sistema de ecuaciones ...,enunciados_ejercicios/Mates CCSS/sistemas_ecua...,enunciados_ejercicios/Mates CCSS/sistemas_ecua...,NaN
2004,Mates II,Mates II,2006,Reserva 2,Ejercicio 4. Opción B.,Geometría,MATEMÁTICAS II. 2006. RESERVA 2. EJERCICIO 4. ..., x yz 1\nConsidera la recta r de ecuacione...,enunciados_ejercicios/Mates II/geometría/Mate...,enunciados_ejercicios/Mates II/geometría/png/...,NaN


Limpiar los enunciados

In [15]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [16]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    # se usa str() para convertir año (int) en string
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [17]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [18]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [19]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
2378,mates_ii_integrales_2021_reserva_3_ejercicio_3,Mates II,Mates II,2021,Reserva 3,Ejercicio 3,Integrales,MATEMÁTICAS II. 2021. RESERVA 3. EJERCICIO 3,"Considera la función f :0, definida por f...",enunciados_ejercicios/Mates II/integrales/Mate...,enunciados_ejercicios/Mates II/integrales/png/...
3808,quimica_equilibrio_quimico_2009_reserva_1_ejer...,Química,Química,2009,Reserva 1,"Ejercicio 3, opción A",Equilibrio Químico,QUÍMICA. 2009. RESERVA 1. EJERCICIO 3. OPCIÓN A,Considere el siguiente sistema general en equi...,enunciados_ejercicios/Química/equilibrio_quí...,enunciados_ejercicios/Química/equilibrio_quí...
856,mates_ccss_inferencia_estadistica_2022_reserva...,Mates CCSS,Mates CCSS,2022,Reserva 4,Ejercicio D7,Inferencia Estadística,SOCIALES II. 2022 RESERVA 4. EJERCICIO D7,Se desea estimar la proporción de personas may...,enunciados_ejercicios/Mates CCSS/inferencia_es...,enunciados_ejercicios/Mates CCSS/inferencia_es...
515,mates_ccss_contraste_de_hipotesis_2010_reserva...,Mates CCSS,Mates CCSS,2010,Reserva 2,Ejercicio 4 opción B,Contraste De Hipótesis,SOCIALES II. 2010 RESERVA 2. EJERCICIO 4 OPCIÓN B,Se sabe que los años de vida de los individuos...,enunciados_ejercicios/Mates CCSS/contraste_hip...,enunciados_ejercicios/Mates CCSS/contraste_hip...
3703,quimica_equilibrio_quimico_2022_reserva_2_ejer...,Química,Química,2022,Reserva 2,Ejercicio C 1,Equilibrio Químico,QUÍMICA. 2022. RESERVA 2. EJERCICIO C 1,"En la reacción: N (g)3H (g) 2NH (g) a 300oC, ...",enunciados_ejercicios/Química/equilibrio_quí...,enunciados_ejercicios/Química/equilibrio_quí...


In [20]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [21]:
df[df.enunciado.isna()]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
442,fisica_ondas_2016_reserva_3_ejercicio_4_opcion_a,Física,Física,2016,Reserva 3,"Ejercicio 4, opción A",Ondas,FISICA. 2016. RESERVA 3. EJERCICIO 4. OPCIÓN A,NaN,enunciados_ejercicios/Física/movimiento_ondul...,enunciados_ejercicios/Física/movimiento_ondul...
4204,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,Química,2026,Junio,Ejercicio 5,Reacciones Redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,NaN,enunciados_ejercicios/Química/redox/Química_...,enunciados_ejercicios/Química/redox/png/png_Q...
4652,quimica_termoquimica_2025_julio_ejercicio_5,Química,Química,2025,Julio,Ejercicio 5,Termoquímica,QUÍMICA.2025. JULIO. EJERCICIO 5,NaN,enunciados_ejercicios/Química/termoquímica/Q...,enunciados_ejercicios/Química/termoquímica/p...
4655,quimica_termoquimica_2025_reserva_2_ejercicio_5,Química,Química,2025,Reserva 2,Ejercicio 5,Termoquímica,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,NaN,enunciados_ejercicios/Química/termoquímica/Q...,enunciados_ejercicios/Química/termoquímica/p...


In [22]:
df.asignatura.unique()

<ArrowStringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [23]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,Campo eléctrico y magnético
121,Física,Campo gravitatorio
244,Física,Física cuántica y nuclear
362,Física,Ondas
449,Física,Óptica geométrica
486,Mates CCSS,Contraste De Hipótesis
545,Mates CCSS,Distribución Binomial
546,Mates CCSS,Distribución Binomial
550,Mates CCSS,Funciones CCSS
806,Mates CCSS,Inferencia Estadística


Arreglar "Distribución binomial "

In [24]:
df.tema = df.tema.str.strip()

In [37]:
df[df.asignatura == "Mates CCSS"].tema.unique()

<ArrowStringArray>
[        'Contraste de Hipótesis',          'Distribución Binomial',
                  'Funciones CCSS',         'Inferencia Estadística',
        'Matrices y Determinantes',                    'Probabilidad',
            'Programación Lineal', 'Sistemas de Ecuaciones Lineales']
Length: 8, dtype: str

In [38]:
df.convocatoria.unique()

<ArrowStringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [40]:
df.loc[df.convocatoria == "Reserva1 Ejercicio",
       ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [41]:
df.to_csv(root/"csv_enunciados/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [40]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [42]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
749,mates_ccss_funciones_ccss_2008_junio_ejercicio...,Mates CCSS,Mates CCSS,2008,Junio,"Ejercicio 2, opción A",Funciones CCSS,SOCIALES II. 2008. JUNIO. EJERCICIO 2. OPCIÓN A, 2x\n si x  2\nSea la función definida de l...,enunciados_ejercicios/Mates CCSS/funciones_ccs...,enunciados_ejercicios/Mates CCSS/funciones_ccs...
4169,quimica_formulacion_2002_junio_ejercicio_1_opc...,Química,Química,2002,Junio,"Ejercicio 1, opción B",Formulación,QUÍMICA. 2002. JUNIO. EJERCICIO 1. OPCIÓN B,Formule o nombre los siguientes compuestos: a)...,enunciados_ejercicios/Química/formulación/Qu...,enunciados_ejercicios/Química/formulación/pn...
3708,quimica_equilibrio_quimico_2021_julio_ejercici...,Química,Química,2021,Julio,Ejercicio B2,Equilibrio Químico,QUÍMICA. 2021. JULIO. EJERCICIO B2,Dado el equilibrio N F (g) 2NF (g) con H0  3...,enunciados_ejercicios/Química/equilibrio_quí...,enunciados_ejercicios/Química/equilibrio_quí...
